# Backfill `recorded_at` with Full Datetime

Many media records have `recorded_at` set to **date-only** (midnight UTC) because the upload form originally used `type="date"` instead of `type="datetime-local"`. This notebook backfills the missing time component.

### Strategy

| Category | Example filename | How we get the time |
|----------|-----------------|--------------------|
| Already has time | N/A | Skip (not midnight) |
| Datetime in filename | `2025_03_08_14_03_35_IMG_5400.MOV` | Parse the `YYYY_MM_DD_HH_MM_SS` prefix |
| Generic filename | `IMG_1234.MOV` | Read MP4/MOV `mvhd` atom from R2 via atom-table walking |

### Safe to re-run

On re-runs, successful R2 extractions are loaded from the previous `backfill_report.csv` — only failures are retried.

### Steps

1. **Connect** to Supabase + R2
2. **Categorize** all records (reuses previous report for R2 results)
3. **Extract** creation_time from R2 for new/failed records only
4. **Review** results + diagnostics on failures
5. **Export** report to CSV
6. **Apply** updates from CSV to Supabase

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

def clean_env(key):
    """Get env var and strip any inline # comments that dotenv may include."""
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

missing = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing:
    print(f"Missing config: {', '.join(missing)}")
else:
    print("All config loaded.")

All config loaded.


In [2]:
import boto3
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

r2 = boto3.client(
    service_name="s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


In [3]:
# ── Fetch all media records ──
import re
from tabulate import tabulate

all_media = []
offset = 0
while True:
    res = sb.table("media").select(
        "id, title, storage_path, original_filename, recorded_at, created_at, file_size_bytes"
    ).range(offset, offset + 1000 - 1).execute()
    all_media.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

print(f"Loaded {len(all_media)} media records.")

Loaded 478 media records.


---
## Step 1: Categorize Records

In [4]:
# ── Categorize by backfill strategy ──
# On re-runs: loads previous backfill_report.csv and reuses successful
# extractions instead of re-fetching from R2. Only retries previous failures.

import csv as csv_mod
from pathlib import Path

REPORT_PATH = Path(r"C:\Users\minds\Videos\DANCE\dance-library\TESTING\backfill_report.csv")

# Matches: 2025_03_08_14_03_35_IMG_5400.MOV  or  VID_20250315_143022.mp4
DATETIME_RE = re.compile(r'(\d{4})[-_](\d{2})[-_](\d{2})[-_](\d{2})[-_](\d{2})[-_](\d{2})')

def is_midnight(recorded_at_str):
    """Check if recorded_at is date-only (midnight UTC)."""
    if not recorded_at_str:
        return True
    return "00:00:00" in recorded_at_str

def extract_datetime_from_filename(filename):
    """Extract full datetime from filename pattern YYYY_MM_DD_HH_MM_SS."""
    if not filename:
        return None
    m = DATETIME_RE.search(filename)
    if m:
        y, mo, d, h, mi, s = m.groups()
        return f"{y}-{mo}-{d}T{h}:{mi}:{s}"
    return None

# ── Load previous report if it exists ──
prev_report = {}  # id -> {status, source, extracted_datetime}
if REPORT_PATH.exists():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        reader = csv_mod.DictReader(f)
        for row in reader:
            prev_report[row["id"]] = row
    prev_ready = sum(1 for r in prev_report.values() if r["status"] == "ready")
    prev_skip = sum(1 for r in prev_report.values() if r["status"] == "skip")
    prev_no_data = sum(1 for r in prev_report.values() if r["status"] == "no_data")
    print(f"Loaded previous report: {len(prev_report)} records")
    print(f"  {prev_ready} ready, {prev_skip} skip, {prev_no_data} no_data (will retry these)")
else:
    print("No previous report found — will process all records from scratch.")

# ── Categorize ──
already_has_time = []     # recorded_at is NOT midnight — skip
from_filename = []        # can extract datetime from filename
from_prev_report = []     # already extracted in a previous run (reuse, don't re-fetch)
needs_r2 = []             # needs MP4 header extraction (new or previously failed)

for r in all_media:
    recorded = r.get("recorded_at")
    filename = r.get("original_filename", "")
    rid = r["id"]
    
    # Already has a real time (not midnight) — already backfilled or never needed it
    if recorded and not is_midnight(recorded):
        already_has_time.append(r)
        continue
    
    # Try extracting datetime from filename
    dt = extract_datetime_from_filename(filename)
    if dt:
        from_filename.append({**r, "extracted_dt": dt})
        continue
    
    # Check if previous report already has a successful R2 extraction for this record
    prev = prev_report.get(rid)
    if prev and prev["status"] == "ready" and prev["source"] == "r2_header" and prev["extracted_datetime"]:
        from_prev_report.append({**r, "extracted_dt": prev["extracted_datetime"]})
        continue
    
    # Needs R2 extraction (new record, or previously failed)
    needs_r2.append(r)

# ── Summary ──
print("\n" + "=" * 60)
print("RECORDED_AT BACKFILL ANALYSIS")
print("=" * 60)
print()

rows = [
    ["Already has time (skip)", len(already_has_time)],
    ["Can extract from filename", len(from_filename)],
    ["Reusing previous R2 extraction", len(from_prev_report)],
    ["Needs R2 extraction (new/retry)", len(needs_r2)],
]
print(tabulate(rows, headers=["Category", "Count"], tablefmt="simple"))
print(f"\nTotal: {len(all_media)}")

Loaded previous report: 478 records
  424 ready, 0 skip, 54 no_data (will retry these)

RECORDED_AT BACKFILL ANALYSIS

Category                           Count
-------------------------------  -------
Already has time (skip)                0
Can extract from filename            133
Reusing previous R2 extraction       291
Needs R2 extraction (new/retry)       54

Total: 478


---
## Step 2: Preview Filename-Based Extractions

In [5]:
print(f"FILENAME-BASED DATETIME EXTRACTIONS ({len(from_filename)} records)")
print(f"{'─' * 80}")
print()

preview = sorted(from_filename, key=lambda x: x["extracted_dt"])[:25]
rows = []
for r in preview:
    rows.append([
        r["original_filename"][:45],
        r.get("recorded_at", "(null)")[:19] if r.get("recorded_at") else "(null)",
        r["extracted_dt"],
    ])
print(tabulate(rows, headers=["Filename", "Current recorded_at", "Extracted datetime"], tablefmt="simple"))

if len(from_filename) > 25:
    print(f"\n  ... and {len(from_filename) - 25} more.")

print(f"\n\nNEEDS R2 EXTRACTION ({len(needs_r2)} records)")
print(f"{'─' * 80}")
print()

preview_r2 = sorted(needs_r2, key=lambda x: x.get("original_filename", ""))[:25]
rows = []
for r in preview_r2:
    rows.append([
        r.get("original_filename", "—")[:40],
        r.get("recorded_at", "(null)")[:19] if r.get("recorded_at") else "(null)",
        r["storage_path"][:45],
    ])
print(tabulate(rows, headers=["Filename", "Current recorded_at", "Storage Path"], tablefmt="simple"))

if len(needs_r2) > 25:
    print(f"\n  ... and {len(needs_r2) - 25} more.")

FILENAME-BASED DATETIME EXTRACTIONS (133 records)
────────────────────────────────────────────────────────────────────────────────

Filename                          Current recorded_at    Extracted datetime
--------------------------------  ---------------------  --------------------
2024_06_02_20_44_34_IMG_0462.MOV  2024-06-02T00:00:00    2024-06-02T20:44:34
2024_06_02_22_49_11_IMG_0463.MOV  2024-06-02T00:00:00    2024-06-02T22:49:11
2024_07_01_21_53_50_IMG_0847.MP4  2024-07-01T00:00:00    2024-07-01T21:53:50
2024_07_04_10_14_14_IMG_0911.MP4  2024-07-04T00:00:00    2024-07-04T10:14:14
2024_07_09_19_54_25_IMG_0975.MOV  2024-07-09T00:00:00    2024-07-09T19:54:25
2024_08_02_12_40_49_IMG_1425.MOV  2024-08-02T00:00:00    2024-08-02T12:40:49
2024_08_02_13_04_19_IMG_1428.MOV  2024-08-02T00:00:00    2024-08-02T13:04:19
2024_08_02_17_16_16_IMG_1430.MP4  2024-08-02T00:00:00    2024-08-02T17:16:16
2024_08_03_13_20_07_IMG_1437.MP4  2024-08-03T00:00:00    2024-08-03T13:20:07
2024_08_09_20_38_37_I

---
## Step 3: MP4/MOV Header Parser

Reads the `mvhd` (movie header) atom from MP4/MOV files to extract `creation_time`.

**How it works:**
1. Read the first 4KB to walk the **top-level atom table** — each atom header is 8 bytes (`size` + `type`), so we can locate the `moov` atom's exact byte offset without downloading the full file
2. Fetch the first **1KB of the `moov` atom** — the `mvhd` sub-atom is always first inside `moov` and only needs ~30 bytes
3. Parse the `creation_time` field (seconds since 1904-01-01 UTC)

This uses **2 small range requests per file** (~4KB + ~1KB = ~5KB total), compared to the old approach of blindly grabbing 128KB chunks that missed files where `moov` was in the middle or in a large trailing section.

In [6]:
import struct
from datetime import datetime, timezone, timedelta

# MP4/MOV epoch: 1904-01-01 00:00:00 UTC
MP4_EPOCH = datetime(1904, 1, 1, tzinfo=timezone.utc)


def parse_mvhd_creation_time(data: bytes) -> str | None:
    """
    Search raw bytes for the 'mvhd' atom and extract creation_time.
    Returns ISO datetime string or None.
    """
    idx = data.find(b'mvhd')
    if idx == -1:
        return None
    
    # mvhd layout: [4 bytes 'mvhd'][1 byte version][3 bytes flags][creation_time]...
    offset = idx + 4  # skip 'mvhd'
    if offset + 8 > len(data):
        return None
    
    version = data[offset]
    offset += 4  # skip version (1) + flags (3)
    
    if version == 0:
        if offset + 4 > len(data):
            return None
        creation_secs = struct.unpack('>I', data[offset:offset+4])[0]
    elif version == 1:
        if offset + 8 > len(data):
            return None
        creation_secs = struct.unpack('>Q', data[offset:offset+8])[0]
    else:
        return None
    
    if creation_secs == 0:
        return None
    
    try:
        creation_dt = MP4_EPOCH + timedelta(seconds=creation_secs)
        if creation_dt.year < 2000 or creation_dt.year > 2030:
            return None
        return creation_dt.strftime('%Y-%m-%dT%H:%M:%S')
    except (OverflowError, OSError):
        return None


def find_moov_offset(data: bytes, file_size: int) -> tuple[int, int] | None:
    """
    Walk top-level MP4/MOV atoms to find moov's (offset, size).
    data should be the first few KB of the file.
    """
    pos = 0
    while pos + 8 <= len(data):
        atom_size = struct.unpack('>I', data[pos:pos+4])[0]
        atom_type = data[pos+4:pos+8]
        
        # Handle extended size (atom_size == 1 means 64-bit size follows)
        if atom_size == 1 and pos + 16 <= len(data):
            atom_size = struct.unpack('>Q', data[pos+8:pos+16])[0]
        
        # atom_size == 0 means "rest of file"
        if atom_size == 0:
            atom_size = file_size - pos
        
        if atom_size < 8:
            break  # corrupt
        
        if atom_type == b'moov':
            return (pos, atom_size)
        
        pos += atom_size
    
    return None


def get_creation_time_from_r2(storage_path: str, file_size: int | None = None) -> str | None:
    """
    Locate moov atom via atom table, then fetch just the mvhd portion.
    Falls back to scanning start/end chunks if atom walking fails.
    """
    # ── Strategy 1: Walk atom table to find moov precisely ──
    if file_size and file_size > 0:
        # Read first 4KB to walk top-level atoms
        try:
            resp = r2.get_object(
                Bucket=R2_BUCKET_NAME,
                Key=storage_path,
                Range="bytes=0-4095"
            )
            header_data = resp['Body'].read()
            
            moov = find_moov_offset(header_data, file_size)
            
            if moov:
                moov_offset, moov_size = moov
                # mvhd is the first sub-atom inside moov, we only need ~200 bytes
                fetch_size = min(1024, moov_size)
                resp = r2.get_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=storage_path,
                    Range=f"bytes={moov_offset}-{moov_offset + fetch_size - 1}"
                )
                moov_data = resp['Body'].read()
                result = parse_mvhd_creation_time(moov_data)
                if result:
                    return result
            
            # moov not in first 4KB of atom headers — it might be near the end.
            # Read the last 4KB to find moov there.
            if file_size > 4096:
                tail_start = max(0, file_size - 4096)
                resp = r2.get_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=storage_path,
                    Range=f"bytes={tail_start}-{file_size - 1}"
                )
                tail_data = resp['Body'].read()
                
                # Walk atoms in the tail to find moov
                # We need to adjust: atoms in tail_data are at absolute offset tail_start
                moov_tail = find_moov_offset(tail_data, file_size - tail_start + tail_start)
                if moov_tail:
                    moov_abs_offset = tail_start + moov_tail[0]
                    fetch_size = min(1024, moov_tail[1])
                    resp = r2.get_object(
                        Bucket=R2_BUCKET_NAME,
                        Key=storage_path,
                        Range=f"bytes={moov_abs_offset}-{moov_abs_offset + fetch_size - 1}"
                    )
                    moov_data = resp['Body'].read()
                    result = parse_mvhd_creation_time(moov_data)
                    if result:
                        return result
        except Exception:
            pass
    
    # ── Strategy 2: Fallback — brute-force scan larger chunks ──
    CHUNK = 512 * 1024  # 512KB
    
    # Try start
    try:
        resp = r2.get_object(
            Bucket=R2_BUCKET_NAME,
            Key=storage_path,
            Range=f"bytes=0-{CHUNK-1}"
        )
        data = resp['Body'].read()
        result = parse_mvhd_creation_time(data)
        if result:
            return result
    except Exception:
        pass
    
    # Try end
    if file_size and file_size > CHUNK:
        try:
            start = file_size - CHUNK
            resp = r2.get_object(
                Bucket=R2_BUCKET_NAME,
                Key=storage_path,
                Range=f"bytes={start}-{file_size-1}"
            )
            data = resp['Body'].read()
            result = parse_mvhd_creation_time(data)
            if result:
                return result
        except Exception:
            pass
    
    return None


print("MP4/MOV header parser ready.")
print(f"{len(needs_r2)} files need R2 extraction.")

MP4/MOV header parser ready.
54 files need R2 extraction.


---
## Step 4: Extract creation_time from R2

Walks the MP4/MOV atom table via ~5KB range requests per file. Falls back to 512KB brute-force chunks if atom walking fails.

In [7]:
r2_results = []  # (record, extracted_datetime_or_None)
extracted_count = 0
failed_count = 0

print(f"Extracting creation_time from {len(needs_r2)} files...")
print()

for i, r in enumerate(needs_r2):
    storage_path = r["storage_path"]
    file_size = r.get("file_size_bytes")
    
    # If file_size is missing from DB, get it from R2
    if not file_size:
        try:
            head = r2.head_object(Bucket=R2_BUCKET_NAME, Key=storage_path)
            file_size = head.get("ContentLength")
        except Exception:
            pass
    
    dt = get_creation_time_from_r2(storage_path, file_size)
    r2_results.append((r, dt))
    
    if dt:
        extracted_count += 1
    else:
        failed_count += 1
    
    if (i + 1) % 50 == 0 or i == len(needs_r2) - 1:
        print(f"  [{i+1}/{len(needs_r2)}] extracted: {extracted_count}, failed: {failed_count}")

print(f"\nDone. Extracted: {extracted_count}, Failed: {failed_count}")

Extracting creation_time from 54 files...

  [50/54] extracted: 0, failed: 50
  [54/54] extracted: 0, failed: 54

Done. Extracted: 0, Failed: 54


---
## Step 5: Review Results

In [8]:
successful = [(r, dt) for r, dt in r2_results if dt]
failed = [(r, dt) for r, dt in r2_results if not dt]

print("=" * 60)
print("R2 EXTRACTION RESULTS")
print("=" * 60)
print(f"  Successfully extracted: {len(successful)}")
print(f"  Failed / no timestamp:  {len(failed)}")

if successful:
    print(f"\n{'─' * 80}")
    print(f"SUCCESSFUL EXTRACTIONS (first 30)")
    print(f"{'─' * 80}")
    rows = []
    for r, dt in sorted(successful, key=lambda x: x[1])[:30]:
        rows.append([
            r.get("original_filename", "—")[:35],
            r.get("recorded_at", "(null)")[:19] if r.get("recorded_at") else "(null)",
            dt,
        ])
    print(tabulate(rows, headers=["Filename", "Current recorded_at", "R2 creation_time"], tablefmt="simple"))
    if len(successful) > 30:
        print(f"  ... and {len(successful) - 30} more.")

if failed:
    print(f"\n{'─' * 80}")
    print(f"FAILED EXTRACTIONS — DIAGNOSTICS ({len(failed)} files)")
    print(f"{'─' * 80}")
    print()

    # Diagnose each failure: is moov missing? Is creation_time = 0? Is moov too deep?
    diag_rows = []
    for r, _ in sorted(failed, key=lambda x: x[0].get("original_filename", "")):
        storage_path = r["storage_path"]
        file_size = r.get("file_size_bytes")
        reason = "unknown"

        if not file_size:
            try:
                head = r2.head_object(Bucket=R2_BUCKET_NAME, Key=storage_path)
                file_size = head.get("ContentLength", 0)
            except Exception:
                reason = "cannot_access_r2"
                diag_rows.append([r.get("original_filename", "—")[:30], reason, "—"])
                continue

        # Check if moov is at the start
        try:
            resp = r2.get_object(Bucket=R2_BUCKET_NAME, Key=storage_path, Range="bytes=0-4095")
            head_data = resp['Body'].read()
            moov = find_moov_offset(head_data, file_size)
            if moov:
                reason = "moov_at_start_but_creation_time_zero"
            else:
                # Check end — read last 64KB to find moov header
                tail_size = min(65536, file_size)
                tail_start = file_size - tail_size
                resp = r2.get_object(Bucket=R2_BUCKET_NAME, Key=storage_path,
                                     Range=f"bytes={tail_start}-{file_size-1}")
                tail_data = resp['Body'].read()

                # Search for 'moov' string in the tail
                moov_idx = tail_data.find(b'moov')
                if moov_idx != -1:
                    # moov found in tail — check if mvhd has creation_time
                    mvhd_idx = tail_data.find(b'mvhd', moov_idx)
                    if mvhd_idx != -1:
                        # Parse the creation_time at this offset
                        off = mvhd_idx + 4
                        if off + 8 <= len(tail_data):
                            ver = tail_data[off]
                            off += 4
                            if ver == 0 and off + 4 <= len(tail_data):
                                ct = struct.unpack('>I', tail_data[off:off+4])[0]
                                reason = f"creation_time=0" if ct == 0 else f"creation_time={ct}_but_out_of_range"
                            elif ver == 1 and off + 8 <= len(tail_data):
                                ct = struct.unpack('>Q', tail_data[off:off+8])[0]
                                reason = f"creation_time=0" if ct == 0 else f"creation_time={ct}_but_out_of_range"
                            else:
                                reason = "mvhd_truncated"
                        else:
                            reason = "mvhd_truncated"
                    else:
                        reason = "moov_found_but_no_mvhd"
                else:
                    # moov not in first 4KB or last 64KB
                    reason = "moov_not_found"
        except Exception as e:
            reason = f"error: {str(e)[:40]}"

        diag_rows.append([
            r.get("original_filename", "—")[:30],
            reason,
            f"{file_size / 1024 / 1024:.1f} MB" if file_size else "—",
        ])

    print(tabulate(diag_rows, headers=["Filename", "Failure Reason", "File Size"], tablefmt="simple"))

    # Summarize reasons
    from collections import Counter
    reason_counts = Counter(row[1] for row in diag_rows)
    print(f"\n{'─' * 40}")
    print("Failure reason summary:")
    for reason, count in reason_counts.most_common():
        print(f"  {reason}: {count}")

R2 EXTRACTION RESULTS
  Successfully extracted: 0
  Failed / no timestamp:  54

────────────────────────────────────────────────────────────────────────────────
FAILED EXTRACTIONS — DIAGNOSTICS (54 files)
────────────────────────────────────────────────────────────────────────────────

Filename      Failure Reason                        File Size
------------  ------------------------------------  -----------
IMG_0617.MP4  moov_at_start_but_creation_time_zero  73.1 MB
IMG_0887.MP4  moov_at_start_but_creation_time_zero  74.4 MB
IMG_0888.MP4  moov_at_start_but_creation_time_zero  60.3 MB
IMG_0895.MOV  moov_at_start_but_creation_time_zero  3.5 MB
IMG_1411.MP4  moov_at_start_but_creation_time_zero  39.0 MB
IMG_1412.MP4  moov_at_start_but_creation_time_zero  17.8 MB
IMG_1415.MP4  moov_at_start_but_creation_time_zero  14.9 MB
IMG_1416.MP4  moov_at_start_but_creation_time_zero  23.3 MB
IMG_1417.MP4  moov_at_start_but_creation_time_zero  36.2 MB
IMG_1418.MP4  moov_at_start_but_creation_time_ze

---
## Step 5: Summary + Export Report to CSV

Saves all results to `backfill_report.csv`. On re-runs, successful extractions are loaded from this file instead of re-fetching from R2.

In [9]:
import csv
from pathlib import Path

# Build lookup of all extracted datetimes
extracted_from_filename = {r["id"]: r["extracted_dt"] for r in from_filename}
extracted_from_r2 = {r["id"]: dt for r, dt in r2_results if dt}
extracted_from_prev = {r["id"]: r["extracted_dt"] for r in from_prev_report}

with open(REPORT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id", "original_filename", "current_recorded_at",
        "source", "extracted_datetime", "status"
    ])
    
    for r in sorted(all_media, key=lambda x: x.get("original_filename", "")):
        rid = r["id"]
        filename = r.get("original_filename", "")
        current = r.get("recorded_at", "")
        
        if rid in extracted_from_filename:
            source = "filename"
            extracted = extracted_from_filename[rid]
            status = "ready"
        elif rid in extracted_from_prev:
            source = "r2_header"
            extracted = extracted_from_prev[rid]
            status = "ready"
        elif rid in extracted_from_r2:
            source = "r2_header"
            extracted = extracted_from_r2[rid]
            status = "ready"
        elif current and not is_midnight(current):
            source = "already_ok"
            extracted = ""
            status = "skip"
        else:
            source = "none"
            extracted = ""
            status = "no_data"
        
        writer.writerow([rid, filename, current, source, extracted, status])

total_ready = len(from_filename) + len(from_prev_report) + len(successful)
total_no_data = len(failed)

print(f"Exported {len(all_media)} rows to: {REPORT_PATH}")
print()
print("=" * 60)
print("BACKFILL SUMMARY")
print("=" * 60)
print()
print(f"  Total records:               {len(all_media)}")
print(f"  Already have time (skip):    {len(already_has_time)}")
print(f"  From filename (ready):       {len(from_filename)}")
print(f"  From previous report (reuse):{len(from_prev_report)}")
print(f"  From R2 header (new):        {len(successful)}")
print(f"  No data available:           {total_no_data}")
print()
print(f"  Total updatable:             {total_ready}")
print(f"  Coverage:                    {(len(already_has_time) + total_ready) / len(all_media) * 100:.1f}%")

Exported 478 rows to: C:\Users\minds\Videos\DANCE\dance-library\TESTING\backfill_report.csv

BACKFILL SUMMARY

  Total records:               478
  Already have time (skip):    0
  From filename (ready):       133
  From previous report (reuse):291
  From R2 header (new):        0
  No data available:           54

  Total updatable:             424
  Coverage:                    88.7%


---
## Step 6: Apply Updates from CSV to Supabase

Reads `backfill_report.csv` and updates all records marked `ready`. Idempotent — skips records that already have a non-midnight time in the DB.

In [10]:
# ── Apply all "ready" records from backfill_report.csv to Supabase ──
# Reads the report directly — no need to re-run extraction cells first.

import csv as csv_mod
from pathlib import Path

report_path = Path(r"C:\Users\minds\Videos\DANCE\dance-library\TESTING\backfill_report.csv")

to_update = []
with open(report_path, "r", encoding="utf-8") as f:
    for row in csv_mod.DictReader(f):
        if row["status"] == "ready" and row["extracted_datetime"]:
            to_update.append(row)

print(f"Found {len(to_update)} records marked 'ready' in report.")
print()

updated = 0
skipped = 0
errors = []
for row in to_update:
    rid = row["id"]
    new_ts = row["extracted_datetime"] + "+00:00"
    
    # Skip if DB already has this time (idempotent)
    current = row.get("current_recorded_at", "")
    if current and "00:00:00" not in current:
        skipped += 1
        continue
    
    try:
        sb.table("media").update({"recorded_at": new_ts}).eq("id", rid).execute()
        updated += 1
    except Exception as e:
        errors.append((rid, row.get("original_filename"), str(e)))

print(f"Updated: {updated}")
print(f"Skipped (already had time): {skipped}")
if errors:
    print(f"\nErrors ({len(errors)}):")
    for rid, fn, err in errors:
        print(f"  {fn} ({rid}): {err}")
else:
    print("No errors.")

Found 424 records marked 'ready' in report.

Updated: 424
Skipped (already had time): 0
No errors.
